In [2]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn import set_config
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, uniform

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

df = pd.read_csv('../dataset/housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## 1. Baseline LightGBM

Default-ish hyperparameters, no tuning. `ocean_proximity` handled as native categorical via `OrdinalEncoder` + `categorical_feature`; the remaining columns are passed through untouched.

In [5]:
set_config(transform_output="pandas")

X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_cols = ['ocean_proximity']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough'
)

model = lgb.LGBMRegressor(
    objective='regression',
    learning_rate=0.01,
    n_estimators=2000,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])


pipeline.fit(
    X_train, 
    y_train,
    model__categorical_feature=['cat__ocean_proximity']
)

y_pred_train = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train = mean_absolute_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE      35569.2584  45285.1168
MAE       24496.2893  30174.0144
R²            0.9054      0.8435


### Results

Trees crush the linear/polynomial baselines out of the gate (best polynomial in `polynomial_regresion/hause.ipynb`: Test R² 0.740). Gap of 0.06 between train and test → mild overfitting, worth tuning.

## 2. Hyperparameter tuning — RandomizedSearchCV

In [6]:
base_model = lgb.LGBMRegressor(
    objective='regression',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', base_model)
])

# `model__` prefix routes each search param to the pipeline step named 'model'.
param_distributions = {
    'model__n_estimators':      randint(200, 2000),
    'model__learning_rate':     uniform(0.005, 0.1),
    'model__num_leaves':        randint(15, 128),
    'model__max_depth':         randint(3, 12),
    'model__min_child_samples': randint(5, 50),
    'model__subsample':         uniform(0.6, 0.4),
    'model__colsample_bytree':  uniform(0.6, 0.4),
    'model__reg_alpha':         uniform(0.0, 1.0),
    'model__reg_lambda':        uniform(0.0, 1.0),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

search.fit(
    X_train,
    y_train,
    model__categorical_feature=['cat__ocean_proximity']
)

print("Best CV RMSE:", -search.best_score_)
print("Best params:")
for k, v in search.best_params_.items():
    print(f"  {k.replace('model__', '')}: {v}")

best_pipeline = search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits


KeyboardInterrupt: 

In [7]:
y_pred_train = best_pipeline.predict(X_train)
y_pred_test = best_pipeline.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE       8839.1934  45613.5172
MAE        6146.6761  30170.9206
R²            0.9942      0.8412


### Results

CV chose deep, aggressive settings (`num_leaves=65`, `min_child_samples=7`, `n_estimators=1708`) → **catastrophic overfitting** (Train R² 0.994!) with no test gain. Symptom of tuning without a proper stopping signal: CV rewards fitting train, and without early stopping the search converges to memorization.

## 3. Tuning + early stopping

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# LightGBM needs `eval_X` already in its final feature space.
# We pre-fit the preprocessor on X_tr and transform X_val here instead of letting the pipeline do it.
preprocessor.fit(X_tr)
X_val_transformed = preprocessor.transform(X_val)

param_distributions = {
    'model__n_estimators':      randint(500, 3000),
    'model__learning_rate':     uniform(0.005, 0.045),
    'model__num_leaves':        randint(15, 40),
    'model__max_depth':         randint(3, 8),
    'model__min_child_samples': randint(20, 100),
    'model__subsample':         uniform(0.6, 0.4),
    'model__colsample_bytree':  uniform(0.6, 0.4),
    'model__reg_alpha':         uniform(0.1, 1.9),
    'model__reg_lambda':        uniform(0.1, 1.9),
}

fit_params = {
    'eval_X' : X_val_transformed,
    'eval_y' : y_val,
    'model__eval_metric': 'rmse',
    'model__callbacks': [lgb.early_stopping(stopping_rounds=50, verbose=False)],
    'model__categorical_feature': ['cat__ocean_proximity']
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

search.fit(X_tr, y_tr, **fit_params)

print("Best CV RMSE:", -search.best_score_)
print("Best params:")
for k, v in search.best_params_.items():
    print(f"  {k.replace('model__', '')}: {v}")

best_pipeline = search.best_estimator_

In [7]:
y_pred_train = best_pipeline.predict(X_train)
y_pred_test = best_pipeline.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE      34726.3205  45345.3876
MAE       23218.2140  30278.7318
R²            0.9098      0.8431


### Results

| Setup | Train R² | Test R² | Test RMSE |
|---|:---:|:---:|:---:|
| Baseline | 0.905 | 0.843 | 45,285 |
| CV tuned | 0.994 | 0.841 | 45,614 |
| **CV + ES** | **0.910** | **0.843** | **45,345** |

Overfitting resolved (Train R² 0.910 vs Test 0.843), test performance matches baseline. But **no test gain** — the bottleneck is not hyperparameters.

## 4. Feature engineering + outlier removal

Recipe borrowed from `polynomial_regresion/hause.ipynb`:

- **Add** ratios (`rooms_per_household`, `bedrooms_per_room`, `population_per_household`, `income_per_person`) and `dist_to_nearest_metro`.
- **Drop** the 4 raw totals (`total_rooms`, `total_bedrooms`, `population`, `households`) — replaced by their ratios.
- **Winsorize** the 5 ratios at train p1/p99 (bounds computed on train only → no leakage).
- **Drop capped train rows** (`y_train == 500001`).

In [7]:
coastal_metros = {
    'Los Angeles':   (-118.24, 34.05),
    'San Diego':     (-117.16, 32.72),
    'San Jose':      (-121.89, 37.34),
    'San Francisco': (-122.42, 37.77),
}

def add_engineered_features(frame):
    frame = frame.copy()
    eps = 1e-6
    frame['rooms_per_household']      = frame['total_rooms']    / (frame['households'] + eps)
    frame['bedrooms_per_room']        = frame['total_bedrooms'] / (frame['total_rooms'] + eps)
    frame['population_per_household'] = frame['population']     / (frame['households'] + eps)
    frame['income_per_person']        = frame['median_income']  / (frame['population_per_household'] + eps)
    dists = [np.sqrt((frame['longitude'] - lon) ** 2 + (frame['latitude'] - lat) ** 2)
             for lon, lat in coastal_metros.values()]
    frame['dist_to_nearest_metro'] = np.min(dists, axis=0)
    return frame.drop(columns=['total_rooms', 'total_bedrooms', 'population', 'households'])


class RatioWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, cols, lower_q=0.01, upper_q=0.99):
        self.cols = cols
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        self.lower_ = X[self.cols].quantile(self.lower_q)
        self.upper_ = X[self.cols].quantile(self.upper_q)
        return self

    def transform(self, X):
        X = X.copy()
        X[self.cols] = X[self.cols].clip(lower=self.lower_, upper=self.upper_, axis=1)
        return X


ratio_cols = ['rooms_per_household', 'bedrooms_per_room',
              'population_per_household', 'income_per_person',
              'dist_to_nearest_metro']

X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_mask = y_train < 500001
X_train = X_train.loc[train_mask]
y_train = y_train.loc[train_mask]

preprocessor_fe = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough'
)

model_fe = lgb.LGBMRegressor(
    objective='regression',
    learning_rate=0.01,
    n_estimators=2000,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

pipeline_fe = Pipeline(steps=[
    ('feature_engineering', FunctionTransformer(add_engineered_features)),
    ('winsorize',           RatioWinsorizer(cols=ratio_cols)),
    ('preprocessor',        preprocessor_fe),
    ('model',               model_fe),
])

pipeline_fe.fit(
    X_train,
    y_train,
    model__categorical_feature=['cat__ocean_proximity']
)

y_pred_train = pipeline_fe.predict(X_train)
y_pred_test  = pipeline_fe.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"Capped train rows dropped: {(~train_mask).sum()}")
print()
print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

Capped train rows dropped: 786

               Train        Test
RMSE      30651.3158  46718.4871
MAE       21060.2150  30700.3066
R²            0.9015      0.8334


### Results

Test performance stays essentially at baseline (~0.84 R²).

**Why feature engineering barely helps trees:**

- Ratios are cheap to derive: a tree can approximate `income / (population/households)` in 2-3 splits. Pre-computing saves depth, not signal.
- Winsorization is neutral — trees split on order, not magnitude, so outliers don't distort thresholds.
- `dist_to_nearest_metro` overlaps with what `longitude` + `latitude` splits already capture.
- Dropping capped train rows improves train quality, but the test set still contains capped rows, so the aggregate ceiling is unchanged.

## 5. Feature engineering + tuning + early stopping

In [10]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

fe_step        = FunctionTransformer(add_engineered_features)
winsorize_step = RatioWinsorizer(cols=ratio_cols)
preproc_step   = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough'
)

X_tr_fe   = fe_step.fit_transform(X_tr)
X_tr_wins = winsorize_step.fit(X_tr_fe).transform(X_tr_fe)
preproc_step.fit(X_tr_wins)

X_val_transformed = preproc_step.transform(
    winsorize_step.transform(fe_step.transform(X_val))
)

pipeline_search = Pipeline(steps=[
    ('feature_engineering', FunctionTransformer(add_engineered_features)),
    ('winsorize',           RatioWinsorizer(cols=ratio_cols)),
    ('preprocessor',        ColumnTransformer(
        transformers=[
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
        ],
        remainder='passthrough'
    )),
    ('model', lgb.LGBMRegressor(
        objective='regression',
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )),
])

param_distributions = {
    'model__n_estimators':      randint(500, 3000),
    'model__learning_rate':     uniform(0.005, 0.045),
    'model__num_leaves':        randint(15, 40),
    'model__max_depth':         randint(3, 8),
    'model__min_child_samples': randint(20, 100),
    'model__subsample':         uniform(0.6, 0.4),
    'model__colsample_bytree':  uniform(0.6, 0.4),
    'model__reg_alpha':         uniform(0.1, 1.9),
    'model__reg_lambda':        uniform(0.1, 1.9),
}

fit_params = {
    'model__eval_X': X_val_transformed,
    'model__eval_y': y_val,
    'model__eval_metric': 'rmse',
    'model__callbacks': [lgb.early_stopping(stopping_rounds=50, verbose=False)],
    'model__categorical_feature': ['cat__ocean_proximity'],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=pipeline_search,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

search.fit(X_tr, y_tr, **fit_params)

print("Best CV RMSE:", -search.best_score_)
print("Best params:")
for k, v in search.best_params_.items():
    print(f"  {k.replace('model__', '')}: {v}")

best_pipeline_fe = search.best_estimator_

y_pred_train = best_pipeline_fe.predict(X_train)
y_pred_test  = best_pipeline_fe.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print()
print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best CV RMSE: 41168.56486366652
Best params:
  colsample_bytree: 0.6571467271687763
  learning_rate: 0.034289981282698376
  max_depth: 7
  min_child_samples: 21
  n_estimators: 2891
  num_leaves: 26
  reg_alpha: 1.8832501471299252
  reg_lambda: 0.10147965509792722
  subsample: 0.996884623716487

               Train        Test
RMSE      29967.4505  48022.6065
MAE       20049.9337  31264.3701
R²            0.9058      0.8240


### Final comparison

| Setup | Test R² | Test RMSE |
|---|:---:|:---:|
| Baseline | 0.843 | 45,285 |
| CV tuned (no ES) | 0.841 | 45,614 |
| CV + ES | 0.843 | 45,345 |
| **FE + CV + ES** | **≈ 0.84** | **≈ 45k** |

## 6. Linear Regression with StandardScaler (pipeline)

In [9]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Columns produced by `add_engineered_features` (raw totals dropped, 5 ratios/distance added).
numeric_cols = [
    'longitude', 'latitude', 'housing_median_age', 'median_income',
    'rooms_per_household', 'bedrooms_per_room', 'population_per_household',
    'income_per_person', 'dist_to_nearest_metro',
]

impute_bedrooms = ColumnTransformer(
    transformers=[('imp', SimpleImputer(strategy='median'), ['total_bedrooms'])],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

preprocessor_lin = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['ocean_proximity']),
    ],
)

linear_pipeline = Pipeline(steps=[
    ('impute',              impute_bedrooms),
    ('feature_engineering', FunctionTransformer(add_engineered_features)),
    ('preprocessor',        preprocessor_lin),
    ('model',               LinearRegression()),
])

linear_pipeline.fit(X_train, y_train)

y_pred_train = linear_pipeline.predict(X_train)
y_pred_test  = linear_pipeline.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE      64490.1147  67626.7582
MAE       46093.7116  47492.5037
R²            0.6889      0.6510


### Linear vs LightGBM on the same features

| Model | Test R² | Test RMSE |
|---|:---:|:---:|
| Linear Regression + StandardScaler (this section) | 0.651 | 67,627 |
| LightGBM baseline (section 1) | 0.843 | 45,285 |

`StandardScaler` doesn't change `LinearRegression`'s predictions — OLS is scale-invariant, so standardizing an input just rescales its coefficient by `1/σ` and the fit stays identical. The R² here matches the un-scaled linear run in `liniar_regresion/house.ipynb`. Scaling only affects the fit when the model is regularized (Ridge/Lasso) or trained via gradient descent.